# 02 — Data Cleaning & Feature Engineering

This notebook performs end-to-end data preparation for the Olist E-Commerce dataset:

1. Load all 9 CSV files
2. Merge into a single unified DataFrame
3. Clean missing values, duplicates, and outliers
4. Engineer business-critical features (including churn-readiness columns)
5. Translate product categories to English
6. Validate the final dataset
7. Export to `data/processed/final_dataset.csv`

## 3.1 Load Data
Load all 9 datasets into separate Pandas DataFrames.

In [2]:
import pandas as pd
import numpy as np
import os

data_dir = '../data/raw/'

# Load all 9 datasets
customers = pd.read_csv(os.path.join(data_dir, 'olist_customers_dataset.csv'))
geolocation = pd.read_csv(os.path.join(data_dir, 'olist_geolocation_dataset.csv'))
order_items = pd.read_csv(os.path.join(data_dir, 'olist_order_items_dataset.csv'))
payments = pd.read_csv(os.path.join(data_dir, 'olist_order_payments_dataset.csv'))
reviews = pd.read_csv(os.path.join(data_dir, 'olist_order_reviews_dataset.csv'))
orders = pd.read_csv(os.path.join(data_dir, 'olist_orders_dataset.csv'))
products = pd.read_csv(os.path.join(data_dir, 'olist_products_dataset.csv'))
sellers = pd.read_csv(os.path.join(data_dir, 'olist_sellers_dataset.csv'))
category_translation = pd.read_csv(os.path.join(data_dir, 'product_category_name_translation.csv'))

print("✅ All 9 datasets loaded successfully.")
for name, frame in [('customers', customers), ('geolocation', geolocation),
                     ('order_items', order_items), ('payments', payments),
                     ('reviews', reviews), ('orders', orders),
                     ('products', products), ('sellers', sellers),
                     ('category_translation', category_translation)]:
    print(f"  {name:>25s}: {frame.shape}")

✅ All 9 datasets loaded successfully.
                  customers: (99441, 5)
                geolocation: (1000163, 5)
                order_items: (112650, 7)
                   payments: (103886, 5)
                    reviews: (99224, 7)
                     orders: (99441, 8)
                   products: (32951, 9)
                    sellers: (3095, 4)
       category_translation: (71, 2)


## 3.2 Merge Data

### Merge Strategy

| Step | Left | Right | Key | How |
|------|------|-------|-----|-----|
| 1 | orders | customers | `customer_id` | left |
| 2 | + order_items | order_items | `order_id` | left |
| 3 | + products | products | `product_id` | left |
| 4 | + sellers | sellers | `seller_id` | left |
| 5 | + reviews | reviews | `order_id` | left |
| 6 | + geolocation (aggregated) | geo_agg | `customer_zip_code_prefix` | left |

**Payments** are aggregated per order *before* merging to avoid row explosion (see § 3.2.2).

**Geolocation** is aggregated to mean lat/lng per zip prefix to avoid duplication bias (see § 3.2.1).

### 3.2.1 Geolocation Aggregation

The raw geolocation table contains **multiple rows per `zip_code_prefix`** (one per street/neighbourhood).  
Simply dropping duplicates would arbitrarily pick one coordinate and distort spatial analysis.

**Approach:** Aggregate latitude and longitude to **mean values per zip prefix**.  
This produces a single representative coordinate per zip area while preserving geographic accuracy.

In [3]:
# Aggregate geolocation: mean lat/lng per zip prefix
geo_agg = (
    geolocation
    .groupby('geolocation_zip_code_prefix', as_index=False)
    .agg(
        geolocation_lat=('geolocation_lat', 'mean'),
        geolocation_lng=('geolocation_lng', 'mean')
    )
)
print(f"Geolocation reduced from {len(geolocation):,} rows → {len(geo_agg):,} unique zip prefixes")

Geolocation reduced from 1,000,163 rows → 19,015 unique zip prefixes


### 3.2.2 Payment Aggregation

The payments table has **multiple rows per order** (e.g. split payments across credit card + voucher).  
Merging raw payments would multiply every order row by its number of payment instalments.

**Approach:** Aggregate per order:
- `total_payment_value` — sum of all payment values for the order
- `payment_installments_total` — max installments across payment methods
- `dominant_payment_type` — the payment method with the highest value for that order

In [4]:
# Aggregate payments per order
payment_value_agg = (
    payments
    .groupby('order_id', as_index=False)
    .agg(
        total_payment_value=('payment_value', 'sum'),
        payment_installments_total=('payment_installments', 'max')
    )
)

# Dominant payment type: the type with the highest value per order
dominant_payment = (
    payments
    .sort_values('payment_value', ascending=False)
    .drop_duplicates(subset='order_id', keep='first')[['order_id', 'payment_type']]
    .rename(columns={'payment_type': 'dominant_payment_type'})
)

payments_agg = pd.merge(payment_value_agg, dominant_payment, on='order_id', how='left')
print(f"Payments reduced from {len(payments):,} rows → {len(payments_agg):,} order-level rows")

Payments reduced from 103,886 rows → 99,440 order-level rows


### 3.2.3 Core Merge Chain

In [5]:
# Step 1: orders ← customers
df = pd.merge(orders, customers, on='customer_id', how='left')

# Step 2: + order_items
df = pd.merge(df, order_items, on='order_id', how='left')

# Step 3: + products
df = pd.merge(df, products, on='product_id', how='left')

# Step 4: + sellers
df = pd.merge(df, sellers, on='seller_id', how='left')

# Step 5: + aggregated payments
df = pd.merge(df, payments_agg, on='order_id', how='left')

# Step 6: + reviews
df = pd.merge(df, reviews, on='order_id', how='left')

# Step 7: + aggregated geolocation
df = pd.merge(
    df, geo_agg,
    left_on='customer_zip_code_prefix',
    right_on='geolocation_zip_code_prefix',
    how='left'
)

print(f"✅ Unified DataFrame shape: {df.shape}")

✅ Unified DataFrame shape: (114092, 41)


## 3.3 Data Cleaning
### 3.3.1 Data Types — Timestamp Conversion
Convert all string-based timestamp columns to proper `datetime` objects.

In [6]:
timestamp_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date',
    'shipping_limit_date',
    'review_creation_date',
    'review_answer_timestamp'
]

for col in timestamp_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

print("✅ Timestamp columns converted to datetime.")
df[timestamp_cols].dtypes

✅ Timestamp columns converted to datetime.


order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
shipping_limit_date              datetime64[us]
review_creation_date             datetime64[us]
review_answer_timestamp          datetime64[us]
dtype: object

### 3.3.2 Missing Values — Identification

In [7]:
# Identify columns with missing values
missing_info = df.isnull().sum()
missing_cols = missing_info[missing_info > 0].sort_values(ascending=False)
missing_pct = (missing_cols / len(df) * 100).round(2)

missing_report = pd.DataFrame({'missing_count': missing_cols, 'missing_pct': missing_pct})
print("Missing values per column:\n")
print(missing_report.to_string())

Missing values per column:

                               missing_count  missing_pct
review_comment_title                  100569        88.15
review_comment_message                 65926        57.78
order_delivered_customer_date           3253         2.85
product_category_name                   2390         2.09
product_name_lenght                     2390         2.09
product_description_lenght              2390         2.09
product_photos_qty                      2390         2.09
order_delivered_carrier_date            1980         1.74
review_score                             961         0.84
review_id                                961         0.84
review_answer_timestamp                  961         0.84
review_creation_date                     961         0.84
product_length_cm                        796         0.70
product_height_cm                        796         0.70
product_width_cm                         796         0.70
product_weight_g                         796

### 3.3.3 Missing Value Treatment

| Column(s) | Treatment | Reasoning |
|-----------|-----------|-----------|
| `order_delivered_customer_date` | **Drop rows** | Delivery time is a key metric; undelivered orders cannot be analysed for logistics performance. |
| `price` | **Drop rows** | Price is a critical financial field; rows without it are unusable. |
| `product_weight_g`, `product_length_cm`, `product_height_cm`, `product_width_cm` | **Median imputation** | Numerical product dimensions; median is robust to outliers and preserves distribution shape. |
| `product_photos_qty` | **Fill with 0** | Missing photo count most likely means no photos were uploaded. |
| `product_name_lenght`, `product_description_lenght` | **Median imputation** | Non-critical descriptive metrics; median preserves central tendency. |
| `review_comment_title`, `review_comment_message` | **Fill with placeholder** | Many customers leave no text; 'No Title'/'No Message' preserves the row for score analysis. |
| `geolocation_lat`, `geolocation_lng` | **Leave as NaN** | Some zip codes have no geolocation match; forcing values would create false spatial data. |

In [8]:
# 1. Drop rows missing critical fields
df = df.dropna(subset=['order_delivered_customer_date', 'price'])

# 2. Impute product physical properties with median
prod_num_cols = ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
for col in prod_num_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# 3. Impute product metadata
if 'product_photos_qty' in df.columns:
    df['product_photos_qty'] = df['product_photos_qty'].fillna(0)
if 'product_name_lenght' in df.columns:
    df['product_name_lenght'] = df['product_name_lenght'].fillna(df['product_name_lenght'].median())
if 'product_description_lenght' in df.columns:
    df['product_description_lenght'] = df['product_description_lenght'].fillna(df['product_description_lenght'].median())

# 4. Impute review text
if 'review_comment_title' in df.columns:
    df['review_comment_title'] = df['review_comment_title'].fillna('No Title')
if 'review_comment_message' in df.columns:
    df['review_comment_message'] = df['review_comment_message'].fillna('No Message')

print(f"✅ Shape after missing-value treatment: {df.shape}")
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]
if len(remaining) > 0:
    print(f"Remaining NaNs (expected — geo/non-critical):\n{remaining}")
else:
    print("No remaining NaN values.")

✅ Shape after missing-value treatment: (110839, 41)
Remaining NaNs (expected — geo/non-critical):
order_approved_at                 15
order_delivered_carrier_date       1
product_category_name           1545
total_payment_value                3
payment_installments_total         3
dominant_payment_type              3
review_id                        827
review_score                     827
review_creation_date             827
review_answer_timestamp          827
geolocation_zip_code_prefix      293
geolocation_lat                  293
geolocation_lng                  293
dtype: int64


### 3.3.4 Duplicates

In [9]:
dup_count = df.duplicated().sum()
print(f"Exact duplicate rows found: {dup_count}")

df = df.drop_duplicates()
print(f"✅ Shape after deduplication: {df.shape}")

Exact duplicate rows found: 0
✅ Shape after deduplication: (110839, 41)


### 3.3.5 Outlier Treatment

#### Why cap at the 99th percentile?

We use the **99th percentile** as the Winsorization threshold because:
1. **Business justification:** In e-commerce, the top 1 % of prices typically represent bulk/B2B orders or luxury items that are not representative of typical consumer behaviour. These extreme values inflate mean-based revenue metrics and distort model training.
2. **Statistical justification:** Winsorizing at the 99th percentile preserves 99 % of the natural distribution while limiting the influence of extreme outliers. This is less aggressive than IQR-based removal (which would discard ≈5 % of data) and retains more information.
3. **Delivery time:** Negative delivery times indicate data entry errors (delivered before purchase). Extremely long delivery times (> 99th pctl) are edge cases (customs delays, re-shipments) that would skew logistics analysis.

We apply the same treatment to `freight_value` since it follows similar skew patterns.

In [10]:
# --- Price ---
price_99 = df['price'].quantile(0.99)
price_01 = df['price'].quantile(0.01)
print(f"Price — 1st pctl: {price_01:.2f}, 99th pctl: {price_99:.2f}")
df['price'] = df['price'].clip(lower=price_01, upper=price_99)

# --- Freight Value ---
freight_99 = df['freight_value'].quantile(0.99)
freight_01 = df['freight_value'].quantile(0.01)
print(f"Freight — 1st pctl: {freight_01:.2f}, 99th pctl: {freight_99:.2f}")
df['freight_value'] = df['freight_value'].clip(lower=freight_01, upper=freight_99)

# --- Delivery Time (temporary column for capping) ---
df['_delivery_days'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.total_seconds() / 86400

neg_delivery = (df['_delivery_days'] < 0).sum()
print(f"Negative delivery times detected (data errors): {neg_delivery}")

delivery_99 = df['_delivery_days'].quantile(0.99)
df['_delivery_days'] = df['_delivery_days'].clip(lower=0, upper=delivery_99)
print(f"Delivery time capped at [0, {delivery_99:.1f}] days")

# Drop temp column — we'll compute final delivery_time in Feature Engineering
df = df.drop(columns=['_delivery_days'])

print("\n✅ Outlier treatment complete.")

Price — 1st pctl: 9.99, 99th pctl: 887.00
Freight — 1st pctl: 4.35, 99th pctl: 83.79
Negative delivery times detected (data errors): 0
Delivery time capped at [0, 45.8] days

✅ Outlier treatment complete.


## 3.4 Feature Engineering

### 3.4.1 Order-Level Features

In [11]:
# order_value = price + freight_value
df['order_value'] = df['price'] + df['freight_value']

# total_order_value: aggregated value per order (across all items in the order)
df['total_order_value'] = df.groupby('order_id')['order_value'].transform('sum')

# delivery_time (in days)
df['delivery_time'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.total_seconds() / 86400

# delay = actual delivery − estimated delivery (positive = late, negative = early)
df['delay'] = (
    df['order_delivered_customer_date'] - df['order_estimated_delivery_date']
).dt.total_seconds() / 86400

print("✅ Order-level features created: order_value, total_order_value, delivery_time, delay")

✅ Order-level features created: order_value, total_order_value, delivery_time, delay


### 3.4.2 Customer-Level Features (Churn Preparation)

These columns are essential for downstream churn / RFM analysis:

| Feature | Definition |
|---------|-----------|
| `customer_total_orders` | Total number of distinct orders placed by the customer |
| `customer_total_spent` | Lifetime revenue from the customer |
| `customer_avg_order_value` | Mean order value per customer |
| `last_purchase_date` | Most recent purchase timestamp |
| `recency` | Days between the dataset's latest date and the customer's last purchase |

In [12]:
# Dataset reference date (max purchase date in the dataset)
dataset_max_date = df['order_purchase_timestamp'].max()
print(f"Dataset reference date (max purchase): {dataset_max_date}")

# Customer-level aggregations
customer_agg = (
    df.groupby('customer_unique_id')
    .agg(
        customer_total_orders=('order_id', 'nunique'),
        customer_total_spent=('total_order_value', 'sum'),
        last_purchase_date=('order_purchase_timestamp', 'max')
    )
    .reset_index()
)

customer_agg['customer_avg_order_value'] = (
    customer_agg['customer_total_spent'] / customer_agg['customer_total_orders']
)

# Recency: days since last purchase relative to dataset end date
customer_agg['recency'] = (
    dataset_max_date - customer_agg['last_purchase_date']
).dt.total_seconds() / 86400

# Merge back into main DataFrame
df = pd.merge(df, customer_agg, on='customer_unique_id', how='left')

print(f"✅ Customer-level features added. Columns: {list(customer_agg.columns)}")
print(f"  Recency range: {df['recency'].min():.1f} – {df['recency'].max():.1f} days")

Dataset reference date (max purchase): 2018-08-29 15:00:37
✅ Customer-level features added. Columns: ['customer_unique_id', 'customer_total_orders', 'customer_total_spent', 'last_purchase_date', 'customer_avg_order_value', 'recency']
  Recency range: 0.0 – 713.1 days


## 3.5 Category Translation
Merge category translation dataset and replace Portuguese names with English.

In [13]:
# Merge translation
df = pd.merge(df, category_translation, on='product_category_name', how='left')

# Fall back to original name if no translation exists, then to 'Unknown'
df['product_category_name_english'] = (
    df['product_category_name_english']
    .fillna(df['product_category_name'])
    .fillna('Unknown')
)

# Drop the original Portuguese column
df = df.drop(columns=['product_category_name'])

print(f"✅ Category translation applied. Unique categories: {df['product_category_name_english'].nunique()}")

✅ Category translation applied. Unique categories: 74


## 3.6 Data Validation

Final sanity checks to ensure the cleaned dataset is consistent and ready for analysis.

In [14]:
print("=" * 60)
print("FINAL DATA VALIDATION REPORT")
print("=" * 60)

# 1. No negative delivery times
neg_dt = (df['delivery_time'] < 0).sum()
print(f"\n1. Negative delivery times:  {neg_dt}  {'✅' if neg_dt == 0 else '❌ INVESTIGATE'}")

# 2. No missing critical fields
critical_fields = ['order_id', 'customer_id', 'price', 'order_delivered_customer_date']
for field in critical_fields:
    nulls = df[field].isnull().sum()
    status = '✅' if nulls == 0 else f'❌ {nulls} nulls'
    print(f"2. Missing {field}: {status}")

# 3. Row count consistency check
print(f"\n3. Final row count: {len(df):,}")
print(f"   Unique orders:  {df['order_id'].nunique():,}")
print(f"   Unique customers: {df['customer_unique_id'].nunique():,}")

# 4. Feature completeness
engineered = ['order_value', 'total_order_value', 'delivery_time', 'delay',
              'customer_total_orders', 'customer_total_spent',
              'customer_avg_order_value', 'last_purchase_date', 'recency',
              'total_payment_value', 'dominant_payment_type']
for feat in engineered:
    present = feat in df.columns
    nulls = df[feat].isnull().sum() if present else 'N/A'
    print(f"4. {feat:>30s}: {'✅ present' if present else '❌ MISSING'}  (nulls: {nulls})")

# 5. Data types
print("\n5. Datetime columns:")
for col in df.select_dtypes(include='datetime64').columns:
    print(f"   {col}: {df[col].dtype}")

print("\n" + "=" * 60)
print("VALIDATION COMPLETE")
print("=" * 60)

FINAL DATA VALIDATION REPORT

1. Negative delivery times:  0  ✅
2. Missing order_id: ✅
2. Missing customer_id: ✅
2. Missing price: ✅
2. Missing order_delivered_customer_date: ✅

3. Final row count: 110,839
   Unique orders:  96,476
   Unique customers: 93,356
4.                    order_value: ✅ present  (nulls: 0)
4.              total_order_value: ✅ present  (nulls: 0)
4.                  delivery_time: ✅ present  (nulls: 0)
4.                          delay: ✅ present  (nulls: 0)
4.          customer_total_orders: ✅ present  (nulls: 0)
4.           customer_total_spent: ✅ present  (nulls: 0)
4.       customer_avg_order_value: ✅ present  (nulls: 0)
4.             last_purchase_date: ✅ present  (nulls: 0)
4.                        recency: ✅ present  (nulls: 0)
4.            total_payment_value: ✅ present  (nulls: 3)
4.          dominant_payment_type: ✅ present  (nulls: 3)

5. Datetime columns:
   order_purchase_timestamp: datetime64[us]
   order_approved_at: datetime64[us]
   order_d

## 3.7 Save Output
Save final dataset to: `data/processed/final_dataset.csv`

In [15]:
import os

os.makedirs('../data/processed', exist_ok=True)

output_path = '../data/processed/final_dataset.csv'
df.to_csv(output_path, index=False)

print(f"✅ Final dataset saved to {output_path}")
print(f"   Shape: {df.shape}")
print(f"   Columns: {list(df.columns)}")

✅ Final dataset saved to ../data/processed/final_dataset.csv
   Shape: (110839, 50)
   Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'total_payment_value', 'payment_installments_total', 'dominant_payment_type', 'review_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp', 'geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'order_value', 'total_order_value', 'deliver